# 05 — LIME Explanations (Group-Level)

Computes Mean Absolute LIME importance aggregated at three levels of granularity
for every run defined in `manifesto_config.pkl`.

| Level | `Group_Variable` | `Group_Value` |
|---|---|---|
| Overall model | `Global` | `Global` |
| By actual-value decile | `Decile` | `D1` … `D10` |
| By segment (or any group_var) | e.g. `Segment` | e.g. `Platinum` |

**LIME strategy**  
LIME is model-agnostic. A `predict_fn` wrapper is built per model family:
- `Linear` — statsmodels formula predict; Segment fixed per row  
- `Tree` — direct sklearn-compatible predict  
- `Neural` — LSTM forward pass with flat-sequence approximation; Segment fixed per row  

Explainer and model artifacts are cached per model name — loaded once, reused across all 7 datasets.

**Sampling**: 1% of rows per Segment (stratified) to keep runtime manageable.

**Output**: `Outputs/lime_importance.csv`  
**No inputs to change.** All configuration is driven by `manifest.csv` via `00_input_manifesto.ipynb`.

In [ ]:
import sys, os, pickle
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from src.shap_utils import load_model_artifacts
from src.lime_utils  import build_lime_explainer, compute_lime_matrix, aggregate_lime

pd.set_option('display.max_columns', 50)
pd.set_option('display.precision', 4)

NOTEBOOK_DIR = os.path.abspath('')
OUTPUT_DIR   = os.path.join(NOTEBOOK_DIR, '..', 'Outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)

SAMPLE_FRAC       = 0.01   # 1% per group for aggregated LIME
MIN_LIME_ROWS     = 20     # floor — tiny OOT buckets still get enough rows for aggregation
MIN_GROUP_SAMPLES = 5

def stratified_sample(df, frac, min_rows, group_var=None, seed=42):
    strat_col = group_var if group_var and group_var in df.columns else None
    if strat_col:
        s = df.groupby(strat_col, group_keys=False).apply(
            lambda x: x.sample(frac=frac, random_state=seed)
        ).reset_index(drop=True)
    else:
        s = df.sample(frac=frac, random_state=seed).reset_index(drop=True)
    if len(s) < min_rows:
        s = df.sample(min(min_rows, len(df)), random_state=seed).reset_index(drop=True)
    return s

print('Libraries loaded')

## 1. Load Manifesto

In [ ]:
with open(os.path.join(OUTPUT_DIR, 'manifesto_config.pkl'), 'rb') as f:
    manifesto = pickle.load(f)

dataset_configs = manifesto['dataset_configs']
group_vars      = manifesto['group_vars']

dev_lookup = {r['model']: r for r in dataset_configs if r['dataset'] == 'DEV'}

print(f'Runs scheduled : {len(dataset_configs)}')
print(f'Group vars     : {group_vars}')

## 2. LIME Pipeline — All Runs

In [ ]:
all_results      = []
artifacts_cache  = {}   # model_name -> (model_obj, features, aux)
explainer_cache  = {}   # model_name -> (explainer, valid_features)
background_cache = {}   # model_name -> background_df

for run in dataset_configs:
    run_id       = run['run_id']
    model_name   = run['model']
    model_family = run['model_family']
    model_type   = run['model_type']
    run_gvars    = run['group_vars']
    run_group_var = run_gvars[0] if run_gvars else None

    print(f'\n[{run_id}]  family={model_family}  type={model_type}')

    # ── Load prediction file (always from manifest path) ──────────────────────
    df = pd.read_csv(run['path'])

    # ── Background: DEV predictions for same model ────────────────────────────
    if model_name not in background_cache:
        dev_run = dev_lookup.get(model_name)
        background_cache[model_name] = pd.read_csv(dev_run['path']) if dev_run else df
    background_df = background_cache[model_name]

    # ── Load model artifacts (cached per model) ───────────────────────────────
    if model_name not in artifacts_cache:
        print(f'  Loading model artifacts ...')
        model_obj, features, aux = load_model_artifacts(run)
        artifacts_cache[model_name] = (model_obj, features, aux)
        print(f'  Features : {len(features)}')

        # Build LIME explainer once per model (features define the explainer)
        print(f'  Building LIME explainer ...')
        explainer, valid_features = build_lime_explainer(background_df, features,
                                                         model_type=model_type)
        explainer_cache[model_name] = (explainer, valid_features)
        print(f'  Valid features for LIME : {len(valid_features)}')
    else:
        model_obj, features, aux = artifacts_cache[model_name]
        explainer, valid_features = explainer_cache[model_name]

    # ── Stratified sample with minimum-rows floor ─────────────────────────────
    sampled_df = stratified_sample(df, SAMPLE_FRAC, MIN_LIME_ROWS,
                                   group_var=run_group_var)
    print(f'  Sampled rows : {len(sampled_df):,} of {len(df):,}')

    # ── Compute LIME weights ──────────────────────────────────────────────────
    lime_arr = compute_lime_matrix(
        sampled_df, explainer, model_obj, model_family, valid_features, aux,
        model_type=model_type, group_var=run_group_var,
    )
    print(f'  LIME array : {lime_arr.shape}')

    # ── Aggregate at Global / Decile / group_vars ─────────────────────────────
    agg_df = aggregate_lime(
        lime_arr, valid_features, sampled_df, df, run,
        run_gvars, min_samples=MIN_GROUP_SAMPLES
    )
    if not agg_df.empty:
        all_results.append(agg_df)
        for gvar, grp in agg_df.groupby('Group_Variable'):
            print(f'  [{gvar}] — {grp["Group_Value"].nunique()} bucket(s)')
    else:
        print(f'  [warn] no groups met min_samples threshold')

print('\nLIME pipeline complete.')

## 3. Save Output

In [ ]:
lime_df = pd.concat(all_results, ignore_index=True)

OUTPUT_COLS = [
    'Dataset', 'Model', 'Model_Domain', 'Model_Type',
    'Group_Variable', 'Group_Value', 'Samples',
    'Feature', 'Mean_Abs_LIME',
]
lime_df = lime_df[[c for c in OUTPUT_COLS if c in lime_df.columns]]
lime_df = lime_df.sort_values(
    ['Dataset', 'Model', 'Group_Variable', 'Group_Value', 'Mean_Abs_LIME'],
    ascending=[True, True, True, True, False]
).reset_index(drop=True)

out_path = os.path.join(OUTPUT_DIR, 'lime_importance.csv')
lime_df.to_csv(out_path, index=False)

n_global = (lime_df['Group_Variable'] == 'Global').sum()
n_decile = (lime_df['Group_Variable'] == 'Decile').sum()
n_group  = (~lime_df['Group_Variable'].isin(['Global', 'Decile'])).sum()
print(f'Saved: lime_importance.csv')
print(f'  Total rows   : {len(lime_df):,}')
print(f'  Global rows  : {n_global:,}')
print(f'  Decile rows  : {n_decile:,}')
print(f'  Group rows   : {n_group:,}')

## 4. Quick Views

In [ ]:
# Top 10 features globally per model (DEV only)
dev_global = lime_df[
    (lime_df['Dataset'] == 'DEV') & (lime_df['Group_Variable'] == 'Global')
]
print('── Top 10 Features by Mean |LIME| — DEV Global ──────────────')
display(
    dev_global.groupby('Model')
    .apply(lambda x: x.nlargest(10, 'Mean_Abs_LIME')[['Feature', 'Mean_Abs_LIME', 'Samples']])
    .reset_index(level=0)
)

In [ ]:
# Segment-level top features (DEV only)
dev_seg = lime_df[
    (lime_df['Dataset'] == 'DEV') & (lime_df['Group_Variable'] == 'Segment')
]
if not dev_seg.empty:
    print('── Top 5 Features per Segment — DEV ─────────────────────────')
    display(
        dev_seg.groupby(['Model', 'Group_Value'])
        .apply(lambda x: x.nlargest(5, 'Mean_Abs_LIME')[['Feature', 'Mean_Abs_LIME']])
        .reset_index(level=[0, 1])
    )